In [5]:
class CurrentTransformerSizing:
    def __init__(self, primary_current, relay_resistance, rated_burden_power, rated_voltage, rated_current_burden,
                 transformer_secondary_power, cable_length, cable_section, short_circuit_current_3ph,
                 short_circuit_current_1ph, winding_resistance, copper_resistivity=0.0175, alpha=0.004, delta_t=70):
        # Electrical parameters
        self.primary_current = primary_current  # A
        self.relay_resistance = relay_resistance  # Ohms
        self.rated_burden_power = rated_burden_power  # VA
        self.rated_voltage = rated_voltage  # V
        self.rated_current_burden = rated_current_burden  # A
        self.transformer_secondary_power = transformer_secondary_power  # VA
        self.cable_length = cable_length  # m
        self.cable_section = cable_section  # mm^2
        self.short_circuit_current_3ph = short_circuit_current_3ph  # A
        self.short_circuit_current_1ph = short_circuit_current_1ph  # A
        self.winding_resistance = winding_resistance  # Ohms

        # Constants
        self.copper_resistivity = copper_resistivity  # Ohm.mm2/m
        self.alpha = alpha  # 1/K
        self.delta_t = delta_t  # °C

    def max_load_current_check(self):
        rated_load_current = self.rated_burden_power / (3 ** 0.5 * self.rated_voltage)
        max_load_current = 1.2 * rated_load_current
        if max_load_current <= self.primary_current:
            return "CT suitable", f"Max load current {max_load_current:.2f} A <= Primary current {self.primary_current} A"
        else:
            return "CT NOT suitable", f"Max load current {max_load_current:.2f} A > Primary current {self.primary_current} A"

    def overcurrent_factor_check(self, kssc=20):
        factor_overcurrent = self.short_circuit_current_3ph / kssc
        if factor_overcurrent <= self.primary_current:
            return "CT suitable", f"Overcurrent factor {factor_overcurrent:.2f} A <= Primary current {self.primary_current} A"
        else:
            return "CT NOT suitable", f"Overcurrent factor {factor_overcurrent:.2f} A > Primary current {self.primary_current} A"

    def cable_resistance(self):
        resistance = (self.copper_resistivity * self.cable_length / self.cable_section) * (1 + self.alpha * self.delta_t)
        return resistance

    def total_secondary_resistance(self):
        return 2 * self.cable_resistance() + self.winding_resistance + self.relay_resistance

    def corrected_overcurrent_factor(self, kssc=20):
        rated_load_resistance = (self.rated_current_burden ** 2) / self.transformer_secondary_power
        kssc2 = kssc * (self.winding_resistance + rated_load_resistance) / self.total_secondary_resistance()
        factor = self.short_circuit_current_3ph / kssc2
        if factor <= self.primary_current:
            return "CT suitable", f"Corrected overcurrent factor {factor:.2f} A <= Primary current {self.primary_current} A"
        else:
            return "CT NOT suitable", f"Corrected overcurrent factor {factor:.2f} A > Primary current {self.primary_current} A"

    def overcurrent_factor_check_1ph(self, kssc=20):
        factor = self.short_circuit_current_1ph / kssc
        if factor <= self.primary_current:
            return "CT suitable", f"1-phase overcurrent factor {factor:.2f} A <= Primary current {self.primary_current} A"
        else:
            return "CT NOT suitable", f"1-phase overcurrent factor {factor:.2f} A > Primary current {self.primary_current} A"

    def corrected_overcurrent_factor_1ph(self, kssc=20):
        rated_load_resistance = (self.rated_current_burden ** 2) / self.transformer_secondary_power
        kssc2 = kssc * (self.winding_resistance + rated_load_resistance) / self.total_secondary_resistance()
        factor = self.short_circuit_current_1ph / kssc2
        if factor <= self.primary_current:
            return "CT suitable", f"Corrected 1-phase overcurrent factor {factor:.2f} A <= Primary current {self.primary_current} A"
        else:
            return "CT NOT suitable", f"Corrected 1-phase overcurrent factor {factor:.2f} A > Primary current {self.primary_current} A"

    def steady_state_check(self, kssc=20):
        rated_load_resistance = (self.rated_current_burden ** 2) / self.transformer_secondary_power
        Vsec_max = kssc * self.rated_current_burden * rated_load_resistance  # V

        Vsec_cc3f = self.short_circuit_current_3ph * (self.total_secondary_resistance()) / self.primary_current
        Vsec_cc1f = self.short_circuit_current_1ph * (self.total_secondary_resistance()) / self.primary_current

        result_3f = "CT suitable" if Vsec_cc3f <= Vsec_max else "CT NOT suitable"
        result_1f = "CT suitable" if Vsec_cc1f <= Vsec_max else "CT NOT suitable"

        reason_3f = f"Vsec_cc3f {Vsec_cc3f:.2f} V {'<=' if Vsec_cc3f <= Vsec_max else '>'} Vsec_max {Vsec_max:.2f} V"
        reason_1f = f"Vsec_cc1f {Vsec_cc1f:.2f} V {'<=' if Vsec_cc1f <= Vsec_max else '>'} Vsec_max {Vsec_max:.2f} V"

        return {"3-phase": (result_3f, reason_3f),
                "1-phase": (result_1f, reason_1f)}


In [6]:
# Instantiate the class
ct = CurrentTransformerSizing(
    primary_current=1600,              # A
    relay_resistance=0.02,             # Ohm
    rated_burden_power=50,             # VA
    rated_voltage=34.5e3,              # V
    rated_current_burden=181.238,      # A
    transformer_secondary_power=50,    # VA
    cable_length=50,                   # m
    cable_section=10,                  # mm^2
    short_circuit_current_3ph=31.5e3,  # A
    short_circuit_current_1ph=31.5e3,  # A
    winding_resistance=0.1             # Ohm
)

# Perform all checks
for check in [
    ct.max_load_current_check,
    ct.overcurrent_factor_check,
    ct.corrected_overcurrent_factor,
    ct.overcurrent_factor_check_1ph,
    ct.corrected_overcurrent_factor_1ph
]:
    status, reason = check()
    print(f"{status}: {reason}")

steady_state_results = ct.steady_state_check()
for phase, (status, reason) in steady_state_results.items():
    print(f"Steady state check ({phase}): {status} - {reason}")


CT suitable: Max load current 0.00 A <= Primary current 1600 A
CT suitable: Overcurrent factor 1575.00 A <= Primary current 1600 A
CT suitable: Corrected overcurrent factor 0.82 A <= Primary current 1600 A
CT suitable: 1-phase overcurrent factor 1575.00 A <= Primary current 1600 A
CT suitable: Corrected 1-phase overcurrent factor 0.82 A <= Primary current 1600 A
Steady state check (3-phase): CT suitable - Vsec_cc3f 6.77 V <= Vsec_max 2381265.25 V
Steady state check (1-phase): CT suitable - Vsec_cc1f 6.77 V <= Vsec_max 2381265.25 V
